<a href="https://colab.research.google.com/github/Freedos1/Cerveau/blob/claude%2Fdetermined-feynman-3o9k1r/Project_Vanishing_Gradient_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project: Exponential Decay of Gradients in RNNs

ALFRED KABORE

This notebook demonstrates, with a fully worked numerical example, why the
gradient of the loss with respect to an early hidden state in a vanilla
(Elman) RNN decays exponentially with the number of time steps between that
state and the loss the vanishing gradient problem and what has to
change for the same recursion to explode instead.



## Part A: System Parameters

We use a standard single-unit (scalar) vanilla RNN cell, the simplest
setting in which the recursive Jacobian structure that causes
vanishing/exploding gradients is fully visible.

**Model equations**

$$z_t = W\,h_{t-1} + U\,x_t + b \qquad\text{(pre-activation)}$$
$$h_t = \tanh(z_t) \qquad\text{(hidden state)}$$

**Parameters chosen for this exercise**

| Symbol | Meaning | Value |
|---|---|---|
| $W$ | recurrent (hidden-to-hidden) weight | 0.9 |
| $U$ | input (input-to-hidden) weight | 0.5 |
| $b$ | bias | 0.1 |
| $x_t$ | input at every time step | 1.0 (constant) |
| $h_0$ | initial hidden state | 0.0 |
| $T$ | number of time steps | 100 |
| $y$ | target value at the final time step | 0.8 |

**Loss function**  mean-squared error evaluated only at the final time
step $T=100$:

$$L = \tfrac{1}{2}\,(y - h_T)^2 \qquad\Longrightarrow\qquad \frac{\partial L}{\partial h_T} = h_T - y$$

We will show that $\partial L/\partial h_1$, the gradient of this single
final loss with respect to the *first* hidden state, is many orders of
magnitude smaller than $\partial L/\partial h_T$ - i.e. it vanishes.

In [ ]:
import math

def tanh(z):
    return math.tanh(z)

# Part A parameters
W, U, b, x, h0, T, y_target = 0.9, 0.5, 0.1, 1.0, 0.0, 100, 0.8

print(f"W={W}  U={U}  b={b}  x_t={x}  h_0={h0}  T={T}  y={y_target}")

W=0.9  U=0.5  b=0.1  x_t=1.0  h_0=0.0  T=100  y=0.8


## Part B: Hidden State Table (t = 1 … 100)

Using the recursion $z_t = W h_{t-1} + Ux + b$, $h_t=\tanh(z_t)$, and the
activation derivative

$$\frac{d}{dz_t}\tanh(z_t) = \operatorname{sech}^2(z_t) = 1-\tanh^2(z_t) = 1-h_t^2$$

we compute $h_t$ and $\operatorname{sech}^2(z_t)$ for every one of the 100
time steps, starting from $h_0=0$. Because $W=0.9<1$ and the input keeps
driving $z_t$ upward, the sequence converges rapidly (by about $t\approx11$)
to the fixed point $h^\*=\tanh(0.9\,h^\*+0.6)\approx0.884494$, at which
point $\operatorname{sech}^2(z_t)$ also settles to a constant
$\approx0.217671$.

In [ ]:
z = [None] * (T + 1)
h = [0.0] * (T + 1)      # h[0] = h_0
dsig = [None] * (T + 1)  # sech^2(z_t) = 1 - h_t^2

for t in range(1, T + 1):
    z[t] = W * h[t - 1] + U * x + b
    h[t] = tanh(z[t])
    dsig[t] = 1 - h[t] ** 2

# ---- full 100-row table ----
print(f"{'t':>4} | {'z_t':>10} | {'h_t = tanh(z_t)':>16} | {'sech^2(z_t) = 1-h_t^2':>22}")
print("-" * 62)
for t in range(1, T + 1):
    print(f"{t:>4} | {z[t]:>10.6f} | {h[t]:>16.6f} | {dsig[t]:>22.6f}")

   t |        z_t |  h_t = tanh(z_t) |  sech^2(z_t) = 1-h_t^2
--------------------------------------------------------------
   1 |   0.600000 |         0.537050 |               0.711578
   2 |   1.083345 |         0.794436 |               0.368871
   3 |   1.314992 |         0.865533 |               0.250852
   4 |   1.378980 |         0.880723 |               0.224328
   5 |   1.392650 |         0.883753 |               0.218981
   6 |   1.395377 |         0.884348 |               0.217928
   7 |   1.395914 |         0.884465 |               0.217721
   8 |   1.396019 |         0.884488 |               0.217681
   9 |   1.396039 |         0.884493 |               0.217673
  10 |   1.396043 |         0.884493 |               0.217671
  11 |   1.396044 |         0.884494 |               0.217671
  12 |   1.396044 |         0.884494 |               0.217671
  13 |   1.396044 |         0.884494 |               0.217671
  14 |   1.396044 |         0.884494 |               0.217671
  15 | 

*(Rows print identically from about $t\approx11$ onward because the
system has converged to its fixed point to within $10^{-6}$; the
underlying values keep approaching the fixed point asymptotically and are
never exactly equal to it. See the full-precision values held in `h`,
`z`, `dsig` above.)*

## Part C: Chain Rule Calculation of $\partial L/\partial h_1$

### Step 1: set up the recursive dependency

Each hidden state depends on the previous one,
$h_k=\tanh(Wh_{k-1}+Ux_k+b)$, so

$$\frac{\partial h_k}{\partial h_{k-1}} = \tanh'(z_k)\cdot W = \operatorname{sech}^2(z_k)\cdot W$$

The loss depends on $h_1$ only through the entire chain
$h_1\to h_2\to h_3\to\dots\to h_T\to L$. By the multivariate chain rule
this dependency is a **product of single-step Jacobians**:

$$\frac{\partial L}{\partial h_1} = \frac{\partial L}{\partial h_T}\cdot\frac{\partial h_T}{\partial h_{T-1}}\cdot\frac{\partial h_{T-1}}{\partial h_{T-2}}\cdots\frac{\partial h_2}{\partial h_1} = \frac{\partial L}{\partial h_T}\prod_{k=2}^{T}\Big(W\cdot\operatorname{sech}^2(z_k)\Big)$$

This is the key formula: $\partial L/\partial h_1$ is $\partial L/\partial
h_T$ multiplied by **99 factors**, each of the form $W\cdot
\operatorname{sech}^2(z_k)$.

### Step 2: evaluate $\partial L/\partial h_T$

From Part A, $L=\tfrac12(y-h_T)^2\Rightarrow \partial L/\partial h_T = h_T-y$.

In [ ]:
dLdhT = h[T] - y_target
print(f"h_T (h_100)     = {h[T]:.10f}")
print(f"dL/dh_T (dL/dh_100) = h_T - y = {dLdhT:.10f}")

h_T (h_100)     = 0.8844936004
dL/dh_T (dL/dh_100) = h_T - y = 0.0844936004


### Step 3: evaluate the product term-by-term

Each factor $W\cdot\operatorname{sech}^2(z_k)$ is $<1$ from the very first
step, so the running product shrinks geometrically. Once the system settles
at its fixed point ($t\gtrsim11$), every remaining factor is essentially
identical, $W\cdot\operatorname{sech}^2(z^\*)=0.9\times0.217671\approx0.195904$,
so for large $k$ the tail of the product behaves like a pure geometric
sequence with ratio $r\approx0.1959$.

In [ ]:
print(f"{'k':>4} | {'sech^2(z_k)':>12} | {'term=W*sech^2':>14} | {'running product':>18}")
print("-" * 58)
prod = 1.0
for t in range(2, 8):
    term = W * dsig[t]
    prod *= term
    print(f"{t:>4} | {dsig[t]:>12.6f} | {term:>14.6f} | {prod:>18.6e}")
print("...")

# full product over all 99 factors, k = 2..100
prod_full = 1.0
for t in range(2, T + 1):
    prod_full *= W * dsig[t]

term_steady = W * dsig[T]
print(f"\nsteady-state term  W*sech^2(z*) = {term_steady:.6f}")
print(f"full product  prod_{{k=2..100}} (W*sech^2(z_k)) = {prod_full:.6e}")

   k |  sech^2(z_k) |  term=W*sech^2 |    running product
----------------------------------------------------------
   2 |     0.368871 |       0.331984 |       3.319841e-01
   3 |     0.250852 |       0.225767 |       7.495107e-02
   4 |     0.224328 |       0.201895 |       1.513224e-02
   5 |     0.218981 |       0.197083 |       2.982309e-03
   6 |     0.217928 |       0.196135 |       5.849357e-04
   7 |     0.217721 |       0.195949 |       1.146177e-04
...

steady-state term  W*sech^2(z*) = 0.195904
full product  prod_{k=2..100} (W*sech^2(z_k)) = 1.656899e-70


### Step 4: combine

$$\frac{\partial L}{\partial h_1} = \frac{\partial L}{\partial h_{100}}\cdot\prod_{k=2}^{100}\big(W\cdot\operatorname{sech}^2(z_k)\big)$$

In [ ]:
dLdh1 = dLdhT * prod_full
print(f"dL/dh_1 = dL/dh_100 * prod_full")
print(f"        = {dLdhT:.6f} * {prod_full:.6e}")
print(f"        = {dLdh1:.6e}")

dL/dh_1 = dL/dh_100 * prod_full
        = 0.084494 * 1.656899e-70
        = 1.399974e-71


**Result:** $\partial L/\partial h_1 \approx 1.400\times10^{-71}$ —
for all practical purposes zero. Even though $\partial L/\partial
h_{100}\approx0.084$ is a perfectly ordinary gradient magnitude, by the
time it is propagated 99 steps back to $h_1$ it has been multiplied by ~99
factors that are each well under 1, and it has collapsed to a number with
71 leading zeros after the decimal point. Gradient-based updates to
whatever parameters influenced $h_1$ (e.g. an embedding used at $t=1$)
would therefore receive essentially no learning signal.This is the
vanishing gradient problem.

## Part D: Analysis: From Vanishing to Exploding

### Why $W=0.9$ vanishes

Every multiplicative factor in the chain, $W\cdot\operatorname{sech}^2(z_k)$,
is bounded above by $W$ (since $\operatorname{sech}^2(z)\le1$ for all $z$,
with equality only at $z=0$). Because $W=0.9<1$, **every single factor in
the product is guaranteed to be less than 1**, so the product is a
strictly decreasing sequence bounded by $0.9^{k-1}$ .It must vanish
geometrically no matter how many steps are taken.

### Re-calculating the first 5 steps with a larger weight

To see the opposite regime we need at least one factor
$W\cdot\operatorname{sech}^2(z_k)>1$, which requires $W>1$ (since
$\operatorname{sech}^2(z_k)\le1$). We recompute the same recursion with the
recurrent weight raised to $W'=1.5$. To isolate the effect of the weight
itself — rather than have the strong constant input $x=1$ immediately drive
$\tanh$ into saturation, which would crush $\operatorname{sech}^2(z_k)$
before the weight's effect could be seen. We set $x_t=0$, $b=0$, keeping a
small nonzero initial state $h_0=0.1$. This keeps $z_t$ near 0 for the
first several steps, where $\operatorname{sech}^2(z_t)\approx1$ and the
recursion is approximately **linear**, $h_t\approx W'h_{t-1}$, so the
product of Jacobians is approximately $(W')^n$.

In [ ]:
def forward_pass(W_, U_, b_, x_, h0_, n):
    hh = [h0_]
    zz = [None]
    for t in range(1, n + 1):
        zt = W_ * hh[t - 1] + U_ * x_ + b_
        zz.append(zt)
        hh.append(tanh(zt))
    return zz, hh

def print_case(label, W_):
    zz, hh = forward_pass(W_, 0.0, 0.0, 0.0, 0.1, 5)
    print(f"-- {label}: W' = {W_} --")
    print(f"{'t':>3} | {'z_t':>10} | {'h_t':>10} | {'sech^2(z_t)':>12} | {'term=W*sech^2':>14} | {'running product':>16}")
    print("-" * 74)
    prod = 1.0
    for t in range(1, 6):
        d_ = 1 - hh[t] ** 2
        term = W_ * d_
        prod *= term
        print(f"{t:>3} | {zz[t]:>10.6f} | {hh[t]:>10.6f} | {d_:>12.6f} | {term:>14.6f} | {prod:>16.6f}")
    print(f"\n=> product of first 5 terms = {prod:.6f}\n")
    return prod

prod5_vanish = print_case("vanishing case", 0.9)
prod5_explode = print_case("exploding case", 1.5)

-- vanishing case: W' = 0.9 --
  t |        z_t |        h_t |  sech^2(z_t) |  term=W*sech^2 |  running product
--------------------------------------------------------------------------
  1 |   0.090000 |   0.089758 |     0.991944 |       0.892749 |         0.892749
  2 |   0.080782 |   0.080607 |     0.993503 |       0.894152 |         0.798254
  3 |   0.072546 |   0.072419 |     0.994755 |       0.895280 |         0.714661
  4 |   0.065177 |   0.065085 |     0.995764 |       0.896188 |         0.640470
  5 |   0.058577 |   0.058510 |     0.996577 |       0.896919 |         0.574450

=> product of first 5 terms = 0.574450

-- exploding case: W' = 1.5 --
  t |        z_t |        h_t |  sech^2(z_t) |  term=W*sech^2 |  running product
--------------------------------------------------------------------------
  1 |   0.150000 |   0.148885 |     0.977833 |       1.466750 |         1.466750
  2 |   0.223328 |   0.219687 |     0.951737 |       1.427606 |         2.093941
  3 |   0.329531 |

### How the result changes from vanishing to exploding

* With $W=0.9$, every term is $<1$, so the running product falls
  monotonically at every step ($0.893\to0.798\to0.715\to0.640\to0.574$,
  $\to0$ as $t\to\infty$). This is **vanishing**: the further back a
  hidden state is, the smaller its influence on the loss gradient,
  exponentially fast.
* With $W'=1.5$, the first four terms are all $>1$ (because
  $\operatorname{sech}^2(z_t)\approx1$ while $h_t$ is still small), so the
  running product **grows** at every step
  ($1.467\to2.094\to2.823\to3.400$) instead of shrinking. This is
  **exploding**: gradients flowing through these early steps are amplified
  rather than attenuated, which in a real network produces huge, unstable
  parameter updates.
* Note the qualitative turning point at $t=5$: as $h_t$ grows toward
  $\pm1$, $\operatorname{sech}^2(z_t)$ starts collapsing toward 0, and the
  5th term ($0.991$) has already dropped back below 1. This illustrates a
  subtlety specific to a *bounded* activation like $\tanh$: because
  $\operatorname{sech}^2(z)\le1$ always, a $\tanh$-RNN cannot sustain true
  long-run exponential explosion — an oversized $W$ can only produce a
  **transient** burst of growth before saturation forces the same
  geometric decay seen in the $W=0.9$ case. (A linear/unbounded recurrence,
  or a $W$ acting on many co-active units as in a real multi-unit RNN, is
  what allows explosion to persist over long horizons — the mechanism
  Pascanu, Mikolov & Bengio, 2013, formalize with the condition
  $\|W\|>1/\max\operatorname{sech}^2(z)=1$ for exploding gradients to even
  be possible.)

### Final values

In [ ]:
print("Final value of dL/dh_1, main system (Parts A-C: W=0.9, x_t=1, b=0.1, h_0=0, T=100):")
print(f"  dL/dh_1 = {dLdh1:.6e}   (vanishing)\n")

print("Product of the first 5 Jacobian terms, Part D comparison (x_t=0, b=0, h_0=0.1):")
print(f"  W  = 0.9  ->  product(first 5 terms) = {prod5_vanish:.6f}   (shrinking, on track to vanish)")
print(f"  W' = 1.5  ->  product(first 5 terms) = {prod5_explode:.6f}   (growing, exploding over this window)")

Final value of dL/dh_1, main system (Parts A-C: W=0.9, x_t=1, b=0.1, h_0=0, T=100):
  dL/dh_1 = 1.399974e-71   (vanishing)

Product of the first 5 Jacobian terms, Part D comparison (x_t=0, b=0, h_0=0.1):
  W  = 0.9  ->  product(first 5 terms) = 0.574450   (shrinking, on track to vanish)
  W' = 1.5  ->  product(first 5 terms) = 3.370723   (growing, exploding over this window)


## Summary

Backpropagation through time multiplies together as many Jacobian factors
$W\cdot\operatorname{sech}^2(z_k)$ as there are time steps separating a
hidden state from the loss. Because $\operatorname{sech}^2(z)\le1$,
whether this product shrinks or grows is governed almost entirely by
whether $|W|<1$ or $|W|>1$:

* $|W|<1\;\Rightarrow$ every factor is $<1$ $\Rightarrow$ the gradient
  **vanishes** geometrically ($\partial L/\partial h_1\approx1.4\times10^{-71}$
  after 99 steps in our example).
* $|W|>1\;\Rightarrow$ some early factors can exceed 1 $\Rightarrow$ the
  gradient can explode over a window of steps, though a bounded
  activation such as $\tanh$ eventually pulls the product back down as the
  hidden state saturates.

This is exactly why vanilla RNNs struggle to learn long-range dependencies,
and why architectures like LSTMs/GRUs (additive, gated state updates) and
techniques like gradient clipping were introduced.